# ClaimsIQ 03 — CrewAI Crew, Connected to Snowflake via MCP

**This notebook:** Rebuilds Lab 19's Intake / Validator / Approver crew,
now with four agents (adding a dedicated Risk Analyst), all pulling real
data from Snowflake through the SAME `mcp_snowflake_server.py` module
Notebook 02 used.

### Prerequisite
Run `00_snowflake_setup_and_seed_data.ipynb` and
`01_mcp_server_snowflake.ipynb` first, in this Jupyter environment.

## Step 1 — Install & import

In [ ]:
%pip install -q crewai crewai-tools openai snowflake-connector-python nest_asyncio

In [ ]:
import os
import nest_asyncio
nest_asyncio.apply()   # required for CrewAI inside Jupyter — see the earlier troubleshooting note

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from mcp_snowflake_server import claims_server, SimpleMCPClient

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before continuing"

mcp_client = SimpleMCPClient(claims_server)
mcp_client.connect()
print("CrewAI connected to the Snowflake MCP server.")

## Step 2 — Wrap each MCP tool for CrewAI

Same `@tool` decorator pattern from Lab 19 — each function is now a
thin wrapper calling `mcp_client.call_tool(...)`, not touching Snowflake
directly. The MCP client is the only thing that knows how to reach the
warehouse.

In [ ]:
@tool("Get Customer Profile")
def get_customer_profile(customer_id: str) -> str:
    """Looks up a customer's profile and recent orders, given a customer ID."""
    return str(mcp_client.call_tool("get_customer_profile", customer_id=customer_id))

@tool("Get Claim Details")
def get_claim_details(claim_id: str) -> str:
    """Looks up a claim's full details joined with its order, given a claim ID."""
    return str(mcp_client.call_tool("get_claim_details", claim_id=claim_id))

@tool("Check Fraud Signals")
def check_fraud_signals(customer_id: str) -> str:
    """Checks fraud signals for a customer in the last 30 days, given a customer ID."""
    return str(mcp_client.call_tool("check_fraud_signals", customer_id=customer_id))

@tool("Get Transaction Velocity")
def get_transaction_velocity(customer_id: str) -> str:
    """Returns a customer's transaction count/total in the last 48 hours, and unrecognized device count."""
    return str(mcp_client.call_tool("get_transaction_velocity", customer_id=customer_id))

@tool("Compare To Peer Spend")
def compare_to_peer_spend(customer_id: str) -> str:
    """Compares a customer's average spend to the overall population's average."""
    return str(mcp_client.call_tool("compare_to_peer_spend", customer_id=customer_id))

print("Five CrewAI-wrapped MCP tools ready.")

## Step 3 — Define four agents

Intake, Risk Analyst (new — owns the fraud-specific tools), Approver.
Notice which tools go to which agent — same “only give what's needed”
discipline from Lab 19.

In [ ]:
intake_agent = Agent(
    role="Claims Intake Specialist",
    goal="Gather customer profile and claim details for a warranty claim",
    backstory="A detail-oriented first point of contact.",
    tools=[get_customer_profile, get_claim_details],
    verbose=True,
)

risk_analyst = Agent(
    role="Risk Analyst",
    goal="Assess fraud risk using transaction velocity, fraud signals, and peer spend comparison",
    backstory="A specialist who distinguishes personal fraud risk from market-wide trends like sale events.",
    tools=[check_fraud_signals, get_transaction_velocity, compare_to_peer_spend],
    verbose=True,
)

claims_approver = Agent(
    role="Claims Approver",
    goal="Make a final APPROVED, DENIED, or ESCALATED decision using the intake summary and risk assessment",
    backstory="A decisive final reviewer who weighs claim plausibility against account risk separately.",
    verbose=True,
)

print("Three agents defined (Intake, Risk Analyst, Approver).")

## Step 4 — Define the tasks

In [ ]:
def build_tasks(customer_id, claim_id):
    intake_task = Task(
        description=f"Gather the customer profile for {customer_id} and claim details for {claim_id}.",
        expected_output="A summary of the customer's profile and the claim's details.",
        agent=intake_agent,
    )
    risk_task = Task(
        description=(
            f"Assess fraud risk for customer {customer_id}: check fraud signals, "
            "transaction velocity in the last 48h, and peer-normalized spend comparison. "
            "Explicitly note whether any spend spike looks personal or market-wide."
        ),
        expected_output="A risk assessment summarizing fraud signals, velocity, and peer comparison, with an explicit personal-vs-market-wide judgment.",
        agent=risk_analyst,
        context=[intake_task],
    )
    approve_task = Task(
        description="Review the intake summary and risk assessment, then decide APPROVED, DENIED, or ESCALATED.",
        expected_output="APPROVED, DENIED, or ESCALATED, with 2-3 sentences of reasoning citing specific evidence.",
        agent=claims_approver,
        context=[intake_task, risk_task],
    )
    return [intake_task, risk_task, approve_task]

print("build_tasks() ready.")

## Step 5 — Run Ananya Rao's case

In [ ]:
tasks = build_tasks("CUST99001", "CLM99001")

crew = Crew(
    agents=[intake_agent, risk_analyst, claims_approver],
    tasks=tasks,
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff()
print("\n" + "=" * 60)
print("FINAL RESULT:")
print(result)

## Deliverable

1. The full `verbose=True` trace and final result.
2. Compare this crew's final decision to Notebook 02's LangGraph
   decision on the SAME case (CUST99001 / CLM99001). Did they agree?
3. One paragraph: the Risk Analyst agent here does the work of THREE
   separate LangGraph executor nodes from Notebook 02
   (executor_fraud handled all three Snowflake calls together). Which
   design would you trust more for a real production system, and why —
   think about what Day 13's tracing would show in each case.